# Utilities testing notebook

The notebook is intended to test the utilities in the `utils` directory, such as dataset loading, preprocessing, and augmentation functions. It will also include examples of how to use these utilities in a typical machine learning workflow.

---
## Setting up paths

In [1]:
import os
import sys
from pathlib import Path

pd = os.getcwd()
utils_path = os.path.join(pd, "..", "utils")

print(f"Current working directory: {pd}")
print(f"Utils path: {utils_path}")

if utils_path not in sys.path:
    sys.path.append(utils_path)

Current working directory: /home/hai/git/WGAN-SOC/notebooks
Utils path: /home/hai/git/WGAN-SOC/notebooks/../utils


## Dataset downloading

- SOFC Microstructures (PFIB-SEM and synthetic) from JPS 2018
- doi: 10.18141/1425617

The dataset is pulled from edx. This is data from the publication "Mesoscale charaterization of local property distribution in hetergeneous electrodes" by Tim Hsu. 

In [2]:
DOWNLOAD_URL = "https://edx.netl.doe.gov/resource/20435741-a7e7-45f7-91a3-e125f8b05b11/download"
OUTPUT_DIR = "data"
FILE_NAME = "edx_doe_data.zip"

In [3]:
from datasets import download_file

final_path = download_file(DOWNLOAD_URL, OUTPUT_DIR, FILE_NAME, extract=True)

File already exists: data/edx_doe_data.zip


Extracting edx_doe_data.zip: 100%|██████████| 127/127 [00:00<00:00, 148.91file/s]

Extracted data/edx_doe_data.zip to data/edx_doe_data


In [4]:
data_path = Path(final_path) / Path(os.listdir(final_path)[0])
os.listdir(data_path)[:5]

['anode_segmented_tiff_z049.tif',
 'anode_segmented_tiff_z080.tif',
 'anode_segmented_tiff_z008.tif',
 'anode_segmented_tiff_z050.tif',
 'anode_segmented_tiff_z062.tif']

## Data Loading

In [5]:
uom_file_path = Path("data/uom/Cell_0A.mat")

In [6]:
from scipy.io import loadmat
uom_data = loadmat(uom_file_path)
uom_data.keys()

dict_keys(['__header__', '__version__', '__globals__', 'vol_seg'])

In [7]:
print(f"Data shape: {uom_data['vol_seg'].shape}")
print(f"Data samples: {uom_data['vol_seg'][:2]}")

Data shape: (400, 1050, 301)
Data samples: [[[2 2 2 ... 1 2 2]
  [2 2 2 ... 1 2 2]
  [2 2 2 ... 1 2 2]
  ...
  [2 2 2 ... 2 2 2]
  [2 2 2 ... 2 2 2]
  [2 2 2 ... 2 2 2]]

 [[2 2 2 ... 1 1 2]
  [2 2 2 ... 1 1 2]
  [2 2 2 ... 1 1 2]
  ...
  [2 2 2 ... 2 2 2]
  [2 2 2 ... 2 2 2]
  [2 2 2 ... 2 2 2]]]


In [8]:
uom_data['vol_seg'].min(), uom_data['vol_seg'].max()

(np.uint8(1), np.uint8(3))

In [9]:
# The mat file is a 3D volume of shape (400, 1050, 301). Each voxel is labeled as either 0 (pores), 1 (Ni), or 2 (YSZ).
# Visualize the data using k3d
import numpy as np
import k3d

# Use uint8 as strictly required by k3d.voxels, avoiding forced memory copies.
# Values 0, 1, and 2 easily fit in uint8 without overflow.
vol_seg = np.asarray(uom_data["vol_seg"], dtype=np.uint8)[...,:5]

# Keep labels in the expected range [0, 2], right now the range is 1 to 3, so we need to shift it down by 1.
vol_seg -= 1

plot = k3d.plot()
plot += k3d.voxels(
    vol_seg,
    color_range=[0, 2],
    opacity=0.5,
)
plot.display()

Output()

In [10]:
vol_seg.min(), vol_seg.max()

(np.uint8(0), np.uint8(2))